In [1]:
import random
import numpy as np
import torch
import pandas as pd
import pyterrier as pt
from pyterrier.measures import *
from collections import defaultdict
import ir_datasets
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import BertTokenizer, BertModel
from tqdm import tqdm

In [2]:
df_colbert = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT E2E/E2E.2019.res.gz")
df_colbert_prf = pt.io.read_results("../ColBERT-PRF-VirtualAppendix/ColBERT-PRF/MSMARCO Passage Res/ColBERT PRF Ranker_beta=1/prf_rank_beta1.2019.res.gz")

In [3]:
def calculate_nqc(df, group_by='qid', score_col='score'):
    """
    计算 NQC (Normalized Query Commitment)
    NQC = std(scores) / mean(scores)
    
    Parameters:
    - df: DataFrame with columns [qid, docno, rank, score, name]
    - group_by: column name to group by (default: 'qid')
    - score_col: column name for scores (default: 'score')
    
    Returns:
    - DataFrame with [qid, nqc]
    """
    nqc_results = []
    
    for qid, group in df.groupby(group_by):
        scores = group[score_col].values
        
        # 计算均值和标准差
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        
        # NQC = std / mean
        if mean_score != 0:
            nqc = std_score / mean_score
        else:
            nqc = 0  # 避免除零
        
        nqc_results.append({
            'qid': qid,
            'nqc': nqc,
            'mean_score': mean_score,
            'std_score': std_score,
            'num_docs': len(scores)
        })
    
    return pd.DataFrame(nqc_results)

# 计算 ColBERT E2E 的 NQC
nqc_colbert = calculate_nqc(df_colbert)
print("ColBERT E2E NQC:")
print(nqc_colbert.head(10))
print(f"\nAverage NQC: {nqc_colbert['nqc'].mean():.4f}")

# 计算 ColBERT PRF 的 NQC
nqc_colbert_prf = calculate_nqc(df_colbert_prf)
print("\n" + "="*50)
print("ColBERT PRF NQC:")
print(nqc_colbert_prf.head(10))
print(f"\nAverage NQC: {nqc_colbert_prf['nqc'].mean():.4f}")

# 比较两者的 NQC
comparison = pd.merge(
    nqc_colbert[['qid', 'nqc']], 
    nqc_colbert_prf[['qid', 'nqc']], 
    on='qid', 
    suffixes=('_e2e', '_prf')
)
comparison['nqc_diff'] = comparison['nqc_prf'] - comparison['nqc_e2e']

print("\n" + "="*50)
print("NQC Comparison (E2E vs PRF):")
print(comparison.head(10))
print(f"\nAverage NQC difference (PRF - E2E): {comparison['nqc_diff'].mean():.4f}")

# 保存结果
nqc_colbert.to_csv('nqc_colbert_e2e.csv', index=False)
nqc_colbert_prf.to_csv('nqc_colbert_prf.csv', index=False)
comparison.to_csv('nqc_comparison.csv', index=False)

print("\n结果已保存到 CSV 文件")

ColBERT E2E NQC:
       qid       nqc  mean_score  std_score  num_docs
0  1037798  0.166602   14.835330   2.471596      4890
1   104861  0.224258   15.753449   3.532840      7291
2  1063750  0.178446    9.752289   1.740253     12482
3  1103812  0.146860   15.668738   2.301106      7778
4  1106007  0.136950   14.759747   2.021347      8705
5  1110199  0.174651   14.427814   2.519825      5434
6  1112341  0.169677   14.848384   2.519427      4772
7  1113437  0.216033   12.324983   2.662604      5266
8  1114646  0.171415   14.031827   2.405265      7493
9  1114819  0.159910   13.716926   2.193479      8608

Average NQC: 0.1757

ColBERT PRF NQC:
       qid       nqc  mean_score  std_score  num_docs
0  1037798  0.065634   34.460859   2.261801      1000
1   104861  0.044069   52.733332   2.323910      1000
2  1063750  0.051817   37.159808   1.925497      1000
3  1103812  0.069327   40.912250   2.836318      1000
4  1106007  0.113890   45.545232   5.187155      1000
5  1110199  0.053174   46.